# 8단계 — 최종 홀드아웃 평가 (설정 고정)

**최종 평가에 앞서 후보 설정을 고정했습니다.**

- 기준: ExtraTrees + LightGBM 스태킹 / LogisticRegression(C=1), 가중치 없음
- **사전 선정 후보:** 동일한 스태킹 / LogisticRegression(C=1), 메타모델 학습 시 `Pastry` 샘플 가중치 **1.50**
- 분할: 앞선 실험과 동일한 `train_test_split(test_size=0.2, stratify=y, random_state=42)`로 **개발 1,552행 / 보류 테스트 389행**
- 메타 학습: 개발 데이터에서만 5-Fold OOF 확률 생성. 테스트 데이터는 학습·가중치 선택·모델 선택에 사용하지 않음
- 두 모델의 **테스트 점수는 투명하게 비교 보고하지만, 테스트 점수로 가중치나 모델을 다시 선택하지 않습니다.**

앞서 개발 데이터에서 가중치와 구조를 여러 번 시험했으므로 개발 CV 점수는 낙관적으로 편향될 수 있습니다. 홀드아웃 테스트 점수 역시 389행 표본의 변동성이 있습니다. 테스트 결과가 목표에 미달하더라도 다시 보정하지 않고 그대로 보고하는 것이 올바른 최종 평가입니다.

필요 파일: `Faults.NNA`, `Faults27x7_var` (노트북과 같은 작업 폴더). 셀을 순서대로 실행하세요.

In [1]:
# 필요한 경우: %pip install lightgbm scikit-learn pandas numpy
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore', category=FutureWarning)
root=Path('.')
for name in ['Faults.NNA','Faults27x7_var']:
    if not (root/name).exists():
        raise FileNotFoundError(f'{name} 파일을 같은 작업 폴더에 두세요. 현재 폴더: {root.resolve()}')
cols=[x.strip() for x in (root/'Faults27x7_var').read_text(encoding='utf-8').splitlines() if x.strip()]
raw=pd.read_csv(root/'Faults.NNA',sep=r'\s+',header=None,names=cols)
assert raw.shape == (1941,34), f'예상 데이터 크기와 다릅니다: {raw.shape}'
encoder=LabelEncoder()
y_full=encoder.fit_transform(raw[cols[27:]].idxmax(axis=1))
X_full=raw.iloc[:,:27].drop(columns='TypeOfSteel_A300')
idx_dev,idx_test=train_test_split(np.arange(len(y_full)),test_size=0.2,stratify=y_full,random_state=42)
X_dev=X_full.iloc[idx_dev].reset_index(drop=True)
y_dev=y_full[idx_dev]
X_test=X_full.iloc[idx_test].reset_index(drop=True)
y_test=y_full[idx_test]
assert len(y_dev)==1552 and len(y_test)==389
print('개발:',X_dev.shape,'| 최종 테스트:',X_test.shape)
print('클래스:',list(encoder.classes_))

개발: (1552, 26) | 최종 테스트: (389, 26)
클래스: ['Bumps', 'Dirtiness', 'K_Scatch', 'Other_Faults', 'Pastry', 'Stains', 'Z_Scratch']


In [2]:
# 7단계와 동일한 베이스모델 설정 (변경 금지)
et_base=ExtraTreesClassifier(
    random_state=42,n_jobs=1,n_estimators=500,
    min_samples_split=5,min_samples_leaf=1,
    max_features=0.8,max_depth=None,
    class_weight='balanced_subsample'
)
lgb_base=LGBMClassifier(
    objective='multiclass',random_state=42,n_jobs=1,verbose=-1,
    subsample_freq=1,subsample=0.8,reg_lambda=0,
    num_leaves=22,n_estimators=700,min_child_samples=5,
    max_depth=-1,learning_rate=0.05,
    colsample_bytree=0.6,class_weight=None
)
inner=StratifiedKFold(n_splits=5,shuffle=True,random_state=1)
print('개발 데이터 OOF 생성: ExtraTrees...',flush=True)
et_oof=cross_val_predict(clone(et_base),X_dev,y_dev,cv=inner,method='predict_proba',n_jobs=1)
print('개발 데이터 OOF 생성: LightGBM...',flush=True)
lgb_oof=cross_val_predict(clone(lgb_base),X_dev,y_dev,cv=inner,method='predict_proba',n_jobs=1)
Z_dev=np.hstack([et_oof,lgb_oof])
print('개발 전체로 베이스 모델 재학습 / 테스트 확률 생성...',flush=True)
et_fit=clone(et_base).fit(X_dev,y_dev)
lgb_fit=clone(lgb_base).fit(X_dev,y_dev)
Z_test=np.hstack([et_fit.predict_proba(X_test),lgb_fit.predict_proba(X_test)])
print('OOF:',Z_dev.shape,'| TEST:',Z_test.shape)

개발 데이터 OOF 생성: ExtraTrees...
개발 데이터 OOF 생성: LightGBM...
개발 전체로 베이스 모델 재학습 / 테스트 확률 생성...
OOF: (1552, 14) | TEST: (389, 14)


In [3]:
# 가중치는 평가 전에 1.50으로 고정. 테스트 결과로 변경하지 않습니다.
pastry_label=int(encoder.transform(['Pastry'])[0])
weight=np.ones(len(y_dev),dtype=float)
weight[y_dev==pastry_label]=1.50

base_meta=LogisticRegression(C=1.0,max_iter=3000)
base_meta.fit(Z_dev,y_dev)
weighted_meta=LogisticRegression(C=1.0,max_iter=3000)
weighted_meta.fit(Z_dev,y_dev,sample_weight=weight)

pred_base=base_meta.predict(Z_test)
pred_weighted=weighted_meta.predict(Z_test)

result=pd.DataFrame([
    {'모델':'기준 스태킹 (LR C=1)',
     '테스트 Macro F1':f1_score(y_test,pred_base,average='macro',zero_division=0),
     'Accuracy':accuracy_score(y_test,pred_base)},
    {'모델':'사전 선정 후보 (Pastry ×1.50)',
     '테스트 Macro F1':f1_score(y_test,pred_weighted,average='macro',zero_division=0),
     'Accuracy':accuracy_score(y_test,pred_weighted)}
])
print('[사전 고정한 두 모델의 최종 홀드아웃 결과]')
display(result.round(4))
diff=float(result.loc[1,'테스트 Macro F1']-result.loc[0,'테스트 Macro F1'])
print(f'후보 - 기준 Macro F1 차이: {diff:+.4f}')
print(f'사전 선정 후보의 0.835 목표 달성 여부: {"달성" if result.loc[1,"테스트 Macro F1"] >= 0.835 else "미달"}')
print('주의: 이 수치를 보고 후보 설정/가중치/시드를 변경하면 독립 테스트의 의미가 사라집니다.')

[사전 고정한 두 모델의 최종 홀드아웃 결과]


,모델,테스트 Macro F1,Accuracy
0,기준 스태킹 (LR C=1),0.8186,0.8046
1,사전 선정 후보 (Pastry ×1.50),0.8164,0.7995


후보 - 기준 Macro F1 차이: -0.0021
사전 선정 후보의 0.835 목표 달성 여부: 미달
주의: 이 수치를 보고 후보 설정/가중치/시드를 변경하면 독립 테스트의 의미가 사라집니다.


In [4]:
# 사전 선정 후보의 클래스별 결과 및 오분류 패턴
labs=np.arange(len(encoder.classes_))
print('[사전 선정 후보: 테스트셋 클래스별 성능]')
print(classification_report(y_test,pred_weighted,labels=labs,
      target_names=encoder.classes_,digits=4,zero_division=0))
print('\n[기준 vs 후보: 클래스별 F1]')
comp=pd.DataFrame({
    '기준':f1_score(y_test,pred_base,average=None,labels=labs,zero_division=0),
    'Pastry ×1.50':f1_score(y_test,pred_weighted,average=None,labels=labs,zero_division=0)
},index=encoder.classes_)
comp['변화']=comp['Pastry ×1.50']-comp['기준']
display(comp.round(4))
print('\n[후보 테스트 오분류 행렬]')
display(pd.DataFrame(confusion_matrix(y_test,pred_weighted,labels=labs),index=encoder.classes_,columns=encoder.classes_))
print('기준 대비 예측이 바뀐 건수:',int(np.sum(pred_base != pred_weighted)))
result.to_csv('stage8_final_test_results.csv',index=False,encoding='utf-8-sig')
print('\n이 결과는 1회 최종 평가로 보고하세요. 같은 테스트셋으로 추가 튜닝하지 마세요.')

[사전 선정 후보: 테스트셋 클래스별 성능]
              precision    recall  f1-score   support

       Bumps     0.7639    0.6790    0.7190        81
   Dirtiness     1.0000    0.8182    0.9000        11
    K_Scatch     0.9730    0.9231    0.9474        78
Other_Faults     0.7143    0.8148    0.7612       135
      Pastry     0.5806    0.5625    0.5714        32
      Stains     0.8571    0.8571    0.8571        14
   Z_Scratch     1.0000    0.9211    0.9589        38

    accuracy                         0.7995       389
   macro avg     0.8413    0.7965    0.8164       389
weighted avg     0.8066    0.7995    0.8008       389


[기준 vs 후보: 클래스별 F1]


,기준,Pastry ×1.50,변화
Bumps,0.7190,0.7190,0.0000
Dirtiness,0.9000,0.9000,0.0000
K_Scatch,0.9474,0.9474,0.0000
Other_Faults,0.7713,0.7612,-0.0101
Pastry,0.5763,0.5714,-0.0048
Stains,0.8571,0.8571,0.0000
Z_Scratch,0.9589,0.9589,0.0000



[후보 테스트 오분류 행렬]


,Bumps,Dirtiness,K_Scatch,Other_Faults,Pastry,Stains,Z_Scratch
Bumps,55,0,0,22,3,1,0
Dirtiness,0,9,0,2,0,0,0
K_Scatch,1,0,72,5,0,0,0
Other_Faults,13,0,1,110,10,1,0
Pastry,2,0,0,12,18,0,0
Stains,1,0,0,1,0,12,0
Z_Scratch,0,0,1,2,0,0,35


기준 대비 예측이 바뀐 건수: 4

이 결과는 1회 최종 평가로 보고하세요. 같은 테스트셋으로 추가 튜닝하지 마세요.
